---
title : "Dam Live"
---

In de toolbox Continu Inzicht wordt DAMlive gebruikt voor het realtime uitvoeren van dijksterkteanalyses. Met DAMlive kunnen voor verschillende scenario’s, op basis van hydraulische belastingen, ondergrondgegevens en de laagopbouw van de dijk, glijcirkels en veiligheidsfactoren worden berekend met behulp van diverse rekenmethodes.

De informatie en output van deze berekeningen worden door DAMlive weggeschreven in een `.stix`-bestand. Dit is een zip-bestand waarin verschillende `.json`-bestanden zijn opgeslagen.

Om de informatie en resultaten weer te geven in Continu Inzicht worden de `.json`-bestanden eerst ingelezen en omgezet naar dataframes ([parsen .json bestanden](#sec-Inladenjson)). Vervolgens worden de dataframes samengevoegd met [StageMerger](#sec-stagemerger), waarna de resultaten uiteindelijk worden geplot ([Plot stages](#sec-plotstages)).

In [ ]:
# initialiseer de (toolbox continu inzicht) modules
from pathlib import Path

from toolbox_continu_inzicht.base.config import Config
from toolbox_continu_inzicht.base.data_adapter import DataAdapter
from toolbox_continu_inzicht.dam_live.combine_results import CombineDamLiveResults

De configuratie ziet er als volgt uit:

````yaml
GlobalVariables:
  rootdir: data_sets/9.dam_live_parse/WV2030_PU0013_87074-1

DataAdapter:
  scenario:
    type: stages
    path: scenarios

  geometries:
    type: geometries
    path: geometries

  soillayers:
    type: soillayers
    path: soillayers

  soils:
    type: soils
    path: soils.json

  waternets:
    type: waternets
    path: waternets

  calculationsettings:
    type: calculationsettings
    path: calculationsettings

  merge_soil:
    type: csv
    path: merged_soil.csv

  merge_waternet:
    type: csv
    path: merged_waternet.csv

  merge_calculations:
    type: csv
    path: merged_calculations.csv
````

In [ ]:
config_path = (
    Path.cwd() / "data_sets" / "9.dam_live_parse" / "dam_live_parse_config.yaml"
)
config = Config(config_path=config_path)
config.lees_config()

adapter = DataAdapter(config)

## Parsen .json-bestanden{#sec-Inladenjson}

Hier worden de verschillende typen `.json`-bestanden ingeladen en omgezet naar een **dataframe** met behulp van een parser. Wanneer er meerdere `.json`-bestanden aanwezig zijn voor een bepaalde outputcategorie, worden deze samengevoegd in één dataframe.  

In het `.stix`-bestand zijn zes typen bestanden te onderscheiden:  

- **Scenario**: Bevat de stage-ID’s van de uitgevoerde berekeningen. Aan de stage-ID’s zijn geometrie, laagopbouw, waterlijnen en berekeningsresultaten gekoppeld via ID’s, die later gebruikt worden voor het mergen.  
- **Geometries**: Hier worden de verschillende lagen van de dijk gedefinieerd met lijnen.  
- **Soillayers**: Koppelt de lagen aan een specifieke grondsoort.  
- **Soils**: Bevat de eigenschappen van alle grondsoorten.  
- **Waternets**: Hier worden de waterlijnen gedefinieerd met punten.  
- **CalculationSettings**: Bevat het berekeningstype en het middelpunt en de radius van de berekende glijcirkels.


In [ ]:
df_stages = adapter.input("scenario")

df_stages

In [ ]:
df_geometries = adapter.input("geometries")

df_geometries.head(5)

In [ ]:
df_soillayers = adapter.input("soillayers")

df_soillayers.head(5)

In [ ]:
df_soils = adapter.input("soils")

df_soils.head(5)

In [ ]:
df_waternets = adapter.input("waternets")

df_waternets.head(5)

In [ ]:
df_calculationsettings = adapter.input("calculationsettings")

df_calculationsettings.head(5)

## StageMerger {#sec-stagemerger}

Vervolgens worden de tabellen aan elkaar gekoppeld. Hierdoor ontstaan tabellen waarin de **stage-ID** wordt gekoppeld aan de informatie uit de **Geometries**, **Soils**, **SoilLayers**, **WaterNets** en **CalculationSettings** tabellen.  

Er worden drie gekoppelde tabellen gemaakt:

- **Soil**: Koppelt de geometrie, laagopbouw en grondtypes aan de stage-ID.  
- **WaterNet**: Koppelt de locaties en typen van de waterlijnen aan de stage-ID.  
- **Calculations**: Koppelt het middelpunt en de radius van de resulterende glijcirkels aan de stage-ID.  

In [ ]:
merge_df = CombineDamLiveResults(data_adapter=adapter)
merge_df.run(
    input=[
        "scenario",
        "geometries",
        "soils",
        "soillayers",
        "waternets",
        "calculationsettings",
    ],
    output=["merge_soil", "merge_waternet", "merge_calculations"],
)

## Plot stage {#sec-plotstage}
Op basis van een opgegeven stage ID en de gekoppelde tabellen, kunnen de dijkopbouw, waterlijnen en resulterende glijcirkels vervolgens geplot worden

In [ ]:
merge_df.plot_stage(
    stage_id=43,
    xlim=(0, 60),
    ylim=(-15, 6),
)